# PBI 7 Random Forest fraud detection

Compare two Random Forest versions for prioritising SP-buy orders for Trust & Safety review. V1 is an unweighted baseline. V2 gives the fraud class more weight to address the missed fraud observed in validation.

## Setup

Select the project Python environment in VS Code and run the cells in order. Put the three historical CSVs under `data/raw/` or set `SPBUY_DATA_DIR`. This notebook reads matching reports under `reports/pbi7-random-forest/` if available and trains when needed. Set `RETRAIN = True` to reproduce the fits or use a different dataset. Each historical fit can take a few minutes.

In [1]:
import json
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from joblib import load

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if not (repo_root / "src" / "models" / "train_random_forest.py").is_file():
    raise FileNotFoundError("Open the notebook from the repository root or notebooks folder.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.models import train_random_forest as rf

THRESHOLD = 0.50  # Provisional until the team agrees on the review threshold.
RETRAIN = False
LOG_TO_MLFLOW = True
EXPERIMENT = "Group2_SPBuy_Fraud_Detection"
REPORT_DIR = repo_root / "reports" / "pbi7-random-forest"

## Data and evaluation

Both versions run the shared cleaners and join the customer, order, and label tables. Only eligible orders with valid fraud labels enter training. Predictors include order value, item count, value per item, order timing, and customer tenure. The target, raw IDs, and current-order refund are excluded from the predictors. The fitted pipeline learns preprocessing on the training rows.

Use the latest slice of training dates to diagnose V1 and assess the V2 change. Keep the later shared evaluation dates for the final comparison. Apply the same threshold to both versions. A false negative is missed fraud and a false positive is a legitimate order flagged for review.

In [2]:
def get_report(split, version):
    report_path = REPORT_DIR / f"{split}-{version}.json"
    model_exists = split != "evaluation" or report_path.with_suffix(".joblib").is_file()

    if not RETRAIN and report_path.is_file() and model_exists:
        report = json.loads(report_path.read_text(encoding="utf-8"))
        if report["threshold"] == THRESHOLD:
            return report

    rf.main([
        "--version", version, "--split", split, "--threshold", str(THRESHOLD),
        "--report-dir", str(REPORT_DIR), "--n-jobs", "4",
    ])
    return json.loads(report_path.read_text(encoding="utf-8"))


def comparison(reports):
    columns = {
        "average_precision": "Average precision",
        "fraud_precision": "Fraud precision",
        "fraud_recall": "Fraud recall",
        "fraud_f1": "Fraud F1",
        "false_negatives": "Missed fraud",
        "false_positives": "False alarms",
    }
    return pd.DataFrame(
        [report["metrics"] for report in reports.values()],
        index=[version.upper() for version in reports],
    )[list(columns)].rename(columns=columns).round(4)

## V1 baseline on validation dates

Fit an unweighted Random Forest with 100 trees, a maximum depth of 12, a minimum of 20 rows per leaf, and a fixed random seed. Check how much fraud the model misses at the provisional threshold.

In [3]:
v1_validation = get_report("validation", "v1")
print(f"Orders in validation: {v1_validation['scored']['rows']:,}")
display(comparison({"v1": v1_validation}))

Orders in validation: 151,542


,Average precision,Fraud precision,Fraud recall,Fraud F1,Missed fraud,False alarms
V1,0.5257,0.736,0.1137,0.197,17622,811


## V2 class weighting on the same validation dates

The baseline missed most labelled fraud cases at 0.50. V2 keeps the other model settings and uses `class_weight="balanced_subsample"` to give fraud cases more weight in each tree. Check whether fewer fraud cases are missed and how many extra orders are flagged.

In [4]:
v2_validation = get_report("validation", "v2")
assert v1_validation["train"] == v2_validation["train"]
assert v1_validation["scored"] == v2_validation["scored"]
display(comparison({"v1": v1_validation, "v2": v2_validation}))

,Average precision,Fraud precision,Fraud recall,Fraud F1,Missed fraud,False alarms
V1,0.5257,0.7360,0.1137,0.1970,17622,811
V2,0.5218,0.3735,0.8001,0.5093,3975,26680


## Evaluate both versions on the holdout

Fit both versions on the full training period and score the same held-out orders. Average precision compares the ranking of orders across thresholds. Fraud precision, recall, F1, and the error counts use the chosen review threshold.

In [5]:
evaluation_reports = {
    version: get_report("evaluation", version)
    for version in ("v1", "v2")
}
v1 = evaluation_reports["v1"]
v2 = evaluation_reports["v2"]
assert v1["train"] == v2["train"] and v1["scored"] == v2["scored"]
assert v1["train"]["end"] < v1["scored"]["start"]

display(pd.DataFrame({"Training": v1["train"], "Evaluation": v1["scored"]}).T)
display(comparison(evaluation_reports))

additional_fraud_found = v2["metrics"]["true_positives"] - v1["metrics"]["true_positives"]
additional_false_alarms = v2["metrics"]["false_positives"] - v1["metrics"]["false_positives"]
print(f"V2 minus V1 at threshold {THRESHOLD:.2f}")
print(f"Fraud orders caught: {additional_fraud_found:+,}")
print(f"Legitimate orders flagged for review: {additional_false_alarms:+,}")

,start,end,rows,fraud_rows
Training,2022-11-12,2023-04-03,758025,80367
Evaluation,2023-04-04,2023-05-11,191847,24055


,Average precision,Fraud precision,Fraud recall,Fraud F1,Missed fraud,False alarms
V1,0.4295,0.5871,0.1877,0.2845,19539,3176
V2,0.4421,0.3204,0.7533,0.4496,5934,38442


V2 minus V1 at threshold 0.50
Fraud orders caught: +13,605
Legitimate orders flagged for review: +35,266


## Log the fitted pipelines and compare in MLflow

Set `MLFLOW_TRACKING_URI` in the local `.env` and enable `LOG_TO_MLFLOW` in Setup when the team server is reachable. This cell uses the saved evaluation pipelines and logs parameters, metrics, the evaluation report, and each fitted model. If a matching run already exists, inspect that run and its model artifacts in Experiments. The two historical fits can be compared under `Group2_SPBuy_Fraud_Detection`.

In [7]:
if LOG_TO_MLFLOW:
    import mlflow
    from config import configure_mlflow

    configure_mlflow()
    experiment = mlflow.set_experiment(EXPERIMENT)
    client = mlflow.MlflowClient()

    for version, report in evaluation_reports.items():
        run_name = f"vivian-random-forest-{version}"
        existing = client.search_runs(
            [experiment.experiment_id], filter_string=f"tags.mlflow.runName = '{run_name}'"
        )
        if existing:
            print(f"{run_name} already exists. Check its model artifacts in MLflow.")
            continue

        model = load(REPORT_DIR / report["model_file"])
        sample = pd.DataFrame(
            [{
                "order_value": 100.0,
                "num_items_ordered": 2.0,
                "value_per_item": 50.0,
                "customer_tenure_days": 45.0,
                "country_code": "SG",
                "collect_type": "delivery",
                "payment_method": "card",
                "order_weekday": "Monday",
                "order_month": "April",
            }],
            columns=report["features"],
        )
        logged = rf.log_run(model, report, sample, model.predict_proba(sample), run_name)
        client.set_tag(logged["run_id"], "review_threshold_status", "provisional")
        print(f"{run_name} logged as run {logged['run_id']}")
else:
    print("Shared MLflow logging is disabled for this notebook run.")

vivian-random-forest-v1 already exists. Check its model artifacts in MLflow.
🏃 View run vivian-random-forest-v2 at: http://group2-mlflow-isabelle.us-east-1.elasticbeanstalk.com/#/experiments/1/runs/c625afee3bf24337b70db5ff4bcf9761
🧪 View experiment at: http://group2-mlflow-isabelle.us-east-1.elasticbeanstalk.com/#/experiments/1


MlflowException: API request to http://group2-mlflow-isabelle.us-east-1.elasticbeanstalk.com/api/2.0/mlflow-artifacts/artifacts/1/models/m-909c51fac2f842f1970bf17ca9d00d95/artifacts/model.pkl failed with timeout exception HTTPConnectionPool(host='group2-mlflow-isabelle.us-east-1.elasticbeanstalk.com', port=80): Max retries exceeded with url: /api/2.0/mlflow-artifacts/artifacts/1/models/m-909c51fac2f842f1970bf17ca9d00d95/artifacts/model.pkl (Caused by ConnectTimeoutError(<HTTPConnection(host='group2-mlflow-isabelle.us-east-1.elasticbeanstalk.com', port=80) at 0x28c51d87790>, 'Connection to group2-mlflow-isabelle.us-east-1.elasticbeanstalk.com timed out. (connect timeout=120)')). To increase the timeout, set the environment variable MLFLOW_HTTP_REQUEST_TIMEOUT (default: 120) to a larger value.